# **Atelier Prompt Engineering**

### `Contexte`

`Une entreprise souhaite mettre en place « AI Business Assistant », un assistant IA polyvalent capable d'aider ses collaborateurs à exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés.`

![contexte](images/assistant.png)

## **Partie 1 – Anatomie d'un prompt**

### Objectif
Construire un prompt structuré, à partir de ses composantes, pour répondre au besoin :
« Je souhaite analyser les retours de clients d'une entreprise. »
 
### Composantes utilisées
 
| **`Composante`** | **`Contenu`** |
|---|---|
| **`Rôle`** | Analyste spécialisé en satisfaction client |
| **`Contexte`** | Une entreprise reçoit des retours clients bruts (avis, messages support, réseaux sociaux) et veut en tirer des enseignements exploitables |
| **`Tâche`** | Analyser un lot de retours clients et en extraire les tendances, problèmes récurrents et points positifs |
| **`Données d'entrée`** | Liste de retours clients (texte brut) |
| **`Contraintes`** | Ne pas inventer d'information absente des retours ; rester factuel ; regrouper les retours similaires ; hiérarchiser par fréquence/gravité |
| **`Format de sortie`** | Structuré en sections : résumé général, points positifs, points négatifs, recommandations |
 
### **Prompt final**

```
<role>
Tu es un analyste spécialisé en satisfaction client et retour d'expérience.
</role>

<contexte> 
Une entreprise collecte des retours clients (avis, messages de support, commentaires)
et souhaite en tirer une analyse exploitable pour améliorer ses produits et services.
</context>

<tache>
Analyse la liste de retours clients fournie ci-dessous. Identifie les thèmes récurrents,
distingue les points positifs des points négatifs, et évalue la fréquence ou la gravité
de chaque problème mentionné.
</tache>

<donnees-entree>
{{liste_des_retours_clients}}
</donnees-entree>

<contraines>
- Ne fournis aucune information qui n'apparaît pas explicitement dans les retours fournis.
- Regroupe les retours similaires sous un même thème plutôt que de les lister un par un.
- Reste factuel et objectif, sans jugement de valeur.
- Priorise les problèmes selon leur fréquence apparente.
</contraintes>

<fornat-de-sortie>
1. Résumé général (2-3 phrases)
2. Points positifs identifiés (liste à puces)
3. Points négatifs / problèmes identifiés (liste à puces, du plus au moins fréquent)
4. Recommandations concrètes (liste à puces)
</format-de-sortie>
```

### **Réponse simulée**

```
1. Résumé général
Les retours indiquent une satisfaction sur la qualité du produit et l'ergonomie
de l'interface, mais révèlent des problèmes techniques récurrents (bugs, plantages)
et des lenteurs au niveau du support et de la livraison.

2. Points positifs identifiés
- Qualité du produit jugée satisfaisante
- Interface simple et agréable à utiliser
- Rapport qualité/prix jugé correct malgré un prix perçu comme élevé

3. Points négatifs / problèmes identifiés (du plus au moins fréquent)
- Bugs et plantages de l'application (paiement, connexion après mise à jour, plantages généraux)
- Retards de livraison sans communication proactive
- Lenteur du support client (délai de réponse de plusieurs jours)

4. Recommandations concrètes
- Prioriser la correction des bugs liés au paiement et à la connexion
- Mettre en place des notifications automatiques en cas de retard de livraison
- Réduire le délai de traitement des tickets de support client
```

## **Partie 2 - Comparison des techniques de prompting**

### **Tâche à tester**

Classer : « Le service est rapide mais l'application plante régulièrement. » → classes possibles : positif, négatif, neutre.

### **Zero-shot**
```
Classe le commentaire suivant en positif, négatif ou neutre :
"Le service est rapide mais l'application plante régulièrement."
```
**Réponse: négatif**

### **One-shot**

```
Voici un exemple de classification :
Commentaire : "Le produit est excellent, je le recommande."
Classe : positif

Classe maintenant ce commentaire :
"Le service est rapide mais l'application plante régulièrement."
Classes possibles : positif, négatif, neutre.
```
**Réponse: négatif**

### **Few-shot**
```
Voici des exemples de classification :
Commentaire : "Le produit est excellent, je le recommande." → positif
Commentaire : "Livraison en retard et aucun geste commercial." → négatif
Commentaire : "Le produit correspond à la description, rien de plus." → neutre

Classe ce commentaire selon les mêmes classes (positif, négatif, neutre) :
"Le service est rapide mais l'application plante régulièrement."
```
**Réponse: négatif**

### **Prompt structuré (rôle + contexte + tâche + contraintes + format)**

```
<role>
Tu es un système de classification de sentiment client.
</role>

<contexte>
Les commentaires peuvent contenir à la fois des aspects positifs et négatifs. 
La classe doit refléter le sentiment global dominant.
</contexte>

<tache>
Classe le commentaire suivant.
"Le service est rapide mais l'application plante régulièrement."
</tache>

<contraintes>
- Choisis une seule classe parmi : positif, négatif, neutre.
- Si le commentaire contient à la fois du positif et du négatif, base ta décision sur 
  l'impact perçu le plus fort pour l'utilisateur (ici, un bug récurrent pèse plus 
  qu'une rapidité de service).
- Justifie ta réponse en une phrase.
</contraintes>

<format-de-sortie>
Classe : <positif|négatif|neutre>
Justification : <une phrase>
</format-de-sortie>
```

### **Classe : négatif**

**Justification** : Le bug récurrent de l'application a un impact négatif plus fort sur l'expérience utilisateur que la rapidité du service, qui reste un point secondaire.

## **Partie 3 – Prompt Engineering et raisonnement**

### **3.1 Décomposition du prompt**

**Prompt à décomposer** :

« Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations. »

```
<role>
     Tu es un analyste customer experience.
</role>

<taches>
     <tache-1>
          Lis les avis clients fournis ci-dessous et identifie les problèmes mentionnés.
     </tache-1>

     <tache-2>
          Classe ces problèmes par ordre d'importance, en te basant sur :
          - leur fréquence d'apparition dans les avis
          - leur impact potentiel sur la satisfaction client
     </tache-2>

     <tache-3>
          Tâche (étape 3) :
          Pour chaque problème classé comme important, propose une recommandation concrète et actionnable.
      </tache-3>
</taches>

<donnees>
     {{avis_clients}}
</donnees>

<contraintes>
     - Ne retiens que les problèmes explicitement mentionnés dans les avis.
     - Une recommandation par problème identifié, pas de recommandation générique.
</contraintes>

<format-de-sortie>
     1. Problèmes classés par importance (avec fréquence estimée)
     2. Recommandations associées à chaque problème
</format-de-sortie>
```

#### **Réponse simulé**

![contexte](images/prompt1_3.1.png)

### **3.2 Prompt de vérification / auto-critique**

```
<role>
     Tu es un analyste customer experience.
</role>

<text-a-analyser>
     "L'application plante souvent au moment du paiement (mentionné dans 5 avis sur 8).
     La livraison est jugée lente par 3 clients. 2 clients complimentent le design de 
     l'interface."
</text-a-analyser>

<tache>
     Identifie les problèmes les plus importants et propose des recommandations.
</tache>

<contraintes>
     - Ne t'appuie que sur les faits présents dans le texte.
     - N'invente aucun chiffre ou avis non mentionné.
</contraintes>
```

**Voici la réponse générée précédemment** :
`{{réponse_du_prompt_1}}`


**Tâche** :
Vérifie cette réponse par rapport au texte source original :
`{{texte_source}}`

**Contrôle spécifiquement** :
1. Informations non justifiées : y a-t-il des affirmations qui ne figurent pas dans le texte source ?
2. Contradictions : la réponse se contredit-elle elle-même ou contredit-elle le texte source ?
3. Informations absentes : des éléments importants du texte source ont-ils été omis ?
4. Hallucinations : des chiffres, faits ou avis ont-ils été inventés ?
5. Respect des contraintes : les règles du prompt initial ont-elles été respectées ?

**Format de sortie** :
- Verdict global : conforme / partiellement conforme / non conforme
- Détail par point de contrôle (1 à 5)
- Corrections proposées si nécessaire

### **Réponse simulée (Prompt 2 — vérification) :**

![](images/prompt_verification3.2.png)